# Πρόβλεψη Καρδιακών Νόσων με χρήση Μηχανικής Μάθησης

Σε αυτή την εργασία θα προσπαθήσουμε να φτιάξουμε μοντέλα που θα προβλέπουν αν ένα άτομο-ασθενής πάσχει από καρδιολογικά προβλήματα βασιζόμενα σε ιατρικά και βιομετρικά τους δεδομένα. Πιο συγκεκριμένα θα φτιάξουμε: 
- ένα μοντέλο Logistic Regression
- ένα Δυαδικό Δέντρο Απόφασης
- ένα μοντέλο Random Forest

In [ ]:
import pandas as pd
data=pd.read_csv('heart.csv')
display(data)
print ('Περιγραφή Δεδομένων:')
display(data.describe())

Παρατηρούμε τα εξής: 
- όλα τα δεδομένα μας ειναι αριθμητικά (θα θέλαμε κάποια κατηγορικά)
- κάποια γνωρίσματα όπως η χοληστερίνη και oldpeak έχουν πολύ μεγάλη διασπορά (ένδειξη για outliers)


Ας ορίσουμε ποιά από τα γνωρίσματα θα θέλαμε για κατηγορικά και ποιά για αριθμητικά για αρχή: 

In [ ]:
categorical_cols = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']
numerical_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']

Θα μπορούσμα να διαλέξουμε να κατηγοριοποιήσουμε την ηλικία των ασθενών φτιάχνοντας beans. Ωστόσο πρέπει να σκεφτούμε αν αυτό θα βοηθούσε τα μοντέλα μας. Τα δεντρα αποφάσεων όπως και τα random forests δεν επωφελούνται από τέτοιου είδους κατηγοριοποιήσεις. Μονάχα η λογιστική παλινδρόμηση θα μπορούσε να δείξει διαφορά μονάχα όμως αν δεν υπήρχε γραμμική συσχέτιση μεταξύ της ηλικίας και του target μας. Ας δούμε λοιπόν πως συσχετίζονται αυτά τα δυο χαρακτηριστικά μαζι: 

In [ ]:
import plotly.express as px
fig = px.histogram(
    data, 
    x="age", 
    color="target", 
    barmode="overlay",      
    title="Κατανομή Ηλικίας ανά Κατάσταση Υγείας",
    labels={"age": "Ηλικία (Έτη)", "count": "Πλήθος"},
    opacity=0.6,            # Διαφάνεια για να φαίνονται οι επικαλύψεις
    marginal="box",         # Προσθέτει boxplot στο πάνω μέρος για σύγκριση διαμέσων
    color_discrete_map={'Υγιείς (0)': '#1f77b4', 'Ασθενείς (1)': '#ff7f0e'} # Χρώματα (μπλε/πορτοκαλί)
)
fig.show(renderer="notebook")

Τα δυο ιστογράμματα έχουν διαφορετικές διαμέσους και μάλιστα οι κατανομή της ηλικίας είναι αρκετά λοξευμένη. (λίγα άτομα ηλικίας $<40$). Επομένως η κατηγοριοποίηση θα μπορούσε μόνο να μπερδέψει το μοντέλο .  Aς ελέγξουμε για διπλότυπα και κενές τιμές:

In [ ]:
print('Διπλότυπα ', data.duplicated().sum())
print('Κενές ', data.isnull().sum())

Δεν έχουμε κενές τιμές επομένως μπορούμε να πετάξουμε απλά τα διπλότυπα.

In [ ]:
data=data.drop_duplicates()

**Παρατήρηση:** Το μεγαλύτερο μέρος από το Dataset μας ήταν διπλότυπα. Η αφαίρεση τους μας αφήνει μονάχα 302 εγγραφές. Αυτό σημαίνει ότι πολύπλοκα μοντέλα όπως τα Random Forests ή τα Decision Trees θα είναι επιρεπή σε overfitting. 

Ας ελέγξουμε τώρα τις υπόλοιπες στήλες για outliers κοιτώντας την λοξότητα του δείγματος και χρησιμοποιώντας boxplots σε αυτές που είναι πολύ λοξές:

In [ ]:
skewness = data[numerical_cols].skew() #ελέγχουμε μόνο της μη κατηγορικές
print("\n--- Λοξότητα (Skewness) ---")
print(skewness)

Μεγαάλη θετική λοξότητα υπάρχει στην χοληστερίνη (chol), πίεση (threstbps) & στην κατάθλιψη (oldpeak) που σημαίνει ότι υπάρχουν κάποιες πολύ μεγάλες τιμές που μας τραβάνε την κατανομή δεξιά από τη μέση τιμή. Μέτρια αρνητική λοξότητα βλέπουμε και στoν μέγιστο καρδιακό παλμό (thalalch). Ας επιβεβαιώσουμε την ύπαρξη outliers με θηκογράμματα: 

In [ ]:
for i, col in enumerate(numerical_cols, 1):
    fig = px.box(
        data, 
        y=col, 
        title=f'Boxplot: {col} (Skewness: {skewness[col]:.2f})',
        points="outliers", 
        width=600, height=400 
    )
    
    # Εμφάνιση
    fig.show(renderer="notebook")

H ύπαρξη outliers μπορεί να επηρεάσει την απόδοση της παλινδρόμησης ωστόσο επειδή πρόκειται για ιατρικά δεδομένα και οι ακραίες τιμές ερμηνεύονται ως ανωμαλίες και ενδείξεις ότι κάτι δεν πάει καλά, επομένως θα πρέπει να τις λάβουμε υπόψη. Για αυτό και θα **εφαρμόσουμε scaling στα δεδομένα μας πριν την εκπαίδευση** κάθε μοντέλου φτιάχνοντας τις λοξές κατανομές. Μένει να ελέγξουμε την κατανομή του target:

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
print("\n--- Κατανομή Target (0: Υγιείς, 1: Ασθενείς) ---")
print(data['target'].value_counts(normalize=True))

# Οπτικοποίηση κατανομής
sns.countplot(x='target', data=data)
plt.title('Κατανομή Καρδιακής Νόσου')
plt.show()

Τώρα μπορούμε να χωρίσουμε το dataset από τα κατηγορικά χαρακτηριστικά με και να ξεχωρίσουμε το target ωστέ να είμαστε έτοιμοι να φτιάξουμε τα μοντέλα supervised learning. H χρήση της get_dummies είναι απαραίτητη ώστε η να γίνει ένα κατάλληλο one-hot encoding που θα μπορεί να χρησιμοποιηθεί από την Lοgistic Regression χωρίς να επιρεαστεί από το μέγεθος των δεικτών/δεδομένων. Τα δενδροειδή μοντέλα θα μπορούσαν να χρησιμοποιήσουν και label encoding

In [ ]:
data = pd.get_dummies(data, columns=categorical_cols, drop_first=True)
X = data.drop('target', axis=1)
y = data['target']

## Κατασκευή - Αξιολόγηση Μοντέλων

Πρωτού φτιάξουμε κάποιο μοντέλο πρέπει να χωρίσουμε το dataset σε train & testing και να εφαρμόσουμε το σωστό scaling. Επειδλη οι κατανομές μας είναι πολύ λοξές θα χρησιμοποιήσουμε **RobustScaler**

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler=RobustScaler()
X_scaled=X.copy()
X_train_scaled=X_train.copy()
X_test_scaled=X_test.copy()
X_scaled[numerical_cols]=scaler.fit_transform(X[numerical_cols])
X_train_scaled[numerical_cols]=scaler.fit_transform(X_train[numerical_cols])
X_test_scaled[numerical_cols] = scaler.transform(X_test[numerical_cols])
print("\n--- Στατιστική Περιγραφή των Scaled Δεδομένων ---")
print(X_scaled.describe())

Βλέποντας τα αποτελέσματα από τον Robust Scaler είμαστε σίγουροι ότι οι κατανομές των χαρακτηριστικών μας δεν είναι πλέον λοξές και μπορούν να χρησιμοποιηθούν για την παλινδρόμηση. 

Για να μπορέσουμε να αξιολογήσουμε τα μοντέλα που θα φτιάξουμε θα βασιστούμε στο πόσο καλή ταξινόμηση κάνουν. Δηλαδή αν προβλέπουν σωστά αν ο ασθενής πάσχει από καρδιακή νόσο. Επομένως θα βασιστούμε στα Classification Reports και Cοnfussion Matrix που παράγονται από κάθε μοντέλο. Στην ουσία ο Confussion Matrix είναι απλά ένας διαγραμματικός τρόπος να δούμε τα αποτελέσματα της ταξινόμησης κατηγοριοποιημένα σε αληθώς θετικά/ψευδή και ψευδώς θετικά/αληθή. Μπορούμε να συμπεριλάβουμε και το f2-score που είναι μια μετρική που συνδιάζει το Precision και το Recall αλλά δίνει περισσότερη βαρύτητα στο Recall. Δεδομένου ότι θέλουμε να είμαστε προληπτικοί και να μετράμε περισσότερα Ψευδώς Θετικούς ασθενείς από Ψευδώς Υγιείς πρέπει να την λάβουμε υπόψη. Μπορούμε λοιπόν να φτιάξουμε μια συνάρτηση που θα συνοψίζει αυτή την διαδικασία:

In [ ]:
from sklearn.metrics import classification_report, fbeta_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
def evaluate_model(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train) 
    y_pred_test = model.predict(X_test)
    y_pred_train = model.predict(X_train)
    y_proba = model.predict_proba(X_test)[:, 1]
    y_proba_train = model.predict_proba(X_train)[:, 1]
    print("Train Classification Report:")
    print(classification_report(y_train, y_pred_train))
    print("Test Classification Report:")
    print(classification_report(y_test, y_pred_test))
    print(f'Test F2 Score: {fbeta_score(y_test, y_pred_test, beta=2):.4f}')
    print(f'Test ROC AUC Score: {roc_auc_score(y_test, y_proba):.4f}')
    cm_test = confusion_matrix(y_test, y_pred_test)
    disp1 = ConfusionMatrixDisplay(confusion_matrix=cm_test, display_labels=model.classes_)
    cm_train = confusion_matrix(y_train, y_pred_train)
    disp2 = ConfusionMatrixDisplay(confusion_matrix=cm_train, display_labels=model.classes_)
    disp1.plot(cmap=plt.cm.Blues)
    plt.title('Confusion Matrix - Test Set')
    plt.show()
    disp2.plot(cmap=plt.cm.Greens)
    plt.title('Confusion Matrix - Train Set')
    plt.show()


Είμαστε τώρα έτοιμοι να ξεκινήσουμε την κατασκευή και αξιολόγηση των μοντέλων μας: 

### Logistic Regression

In [ ]:
model=LogisticRegression()
evaluate_model(model, X_train_scaled, X_test_scaled, y_train, y_test)

Παρατηρούμε ότι το μοντέλο μας έχει αρκετά μεγάλο ποσοστό σε recall. Το γεγονός ότι οι τιμές τις μετρικής accuracy δεν διαφέρουν πολύ σε training & testing set είναι καθυσυχαστικό και σημαίνει ότι το μοντέλο μας δεν έχει υπερπροσαρμοστεί. To precision επίσης είναι υψηλό για την κλάση 1 (άρρωστοι). Πιο συγκεκριμένα έχουμε $Precision_1=0.88$ που σημαίνει πως $88\%$ από τους ασθενείς που προβλέψαμε  πάσχουν όντως από καρδιακή νόσο. Επίσης το $F_2score=85\%$, ενώ το εμβαδόν κάτω από τη καμπύλη ROC είναι περίπου ίσο με 0.9. Από τον Confussion Matrix φαίνεται πως χάσαμε μόνο 5 πραγματικούς ασθενής. Με βάση αυτές τις μετρικές μπορούμε να πούμε ότι το μοντέλο έχει καλή συμπεριφορά 

Ένα πλεονέκτημα που έχει η Λογιστική παλινδρόμηση έναντι των άλλων μοντέλων είναι η ερμηνευσιμότητα της. Κοιτώντας το μέγεθος των συντελεστών μπορούμε να δούμε ποια γνωρίσματα έχουν περισσότερη σημασία για την πρόβλεψη μας. 

In [ ]:
print("\n Logistic Regression Coefficients:")
for feature, coef in zip(X.columns, model.coef_[0]):
    print(f"{feature}: {coef:.4f}") 

Από ότι φαίνεται τα γνωρίσματα που παίζουν το σημαντικότερο ρόλο και έιναι ισχυρές ενδείξεις για καρδιακή νόσο είναι: o μέγιστος καρδιακός παλμός (tachalch), o δείκτης cp που δηλώνει τον τύπο θωρακικού πόνου (δηλαδή οι μεγάλοι θετικοί συντελεστές). 
Αντίστοιχα πολύ αρνητικοί συντελεστές είναι ισχυρές ενδείξεις πως ο ασθενής είναι υγιής (ανδρικό φύλλο, θαλασαιμία τύπου 3, ο δείκτης ca που αναφέρεται στα αγγεία )

### Decision Tree

Η αξιολόγηση ενός decision tree είναι λίγο πιο πολύπλοκη καθώς το μοντέλο μας εξαρτάται από παραμέτρους όπως το μέγιστο βάθος και ο ελάχιστος αριθμός φύλλων. Για να είμαστε σίγουροι ότι έχουμε διαλέξει τις κατάλληλες παραμέτρους δηλαδή τις βέλτιστες προς μεγιστοποίηση της ακρίβειας του μοντέλου μπορούμε να χρησιμοποιήσουμε την **GridSearch**. Όπως είπαμε και πριν πρέπει να είμαστε προσεκτικόι ώστε το δέντρο μας να μην υπερπροσαρμοστεί. Για αυτό και θα προσπαθήσουμε να περιορίσουμε το βάθος του.

In [ ]:
dt_params = {'max_depth': [3, 4, 5], 'min_samples_split': [ 2, 3, 4, 5, 6, 7, 8, 10]}
dt_grid = GridSearchCV(DecisionTreeClassifier(random_state=42), dt_params, cv=5)
dt_grid.fit(X_train, y_train) # Τα δέντρα δεν απαιτούν scaling, άρα χρησιμοποιούμε X_train
best_dt = dt_grid.best_estimator_
print(f"\nBest Decision Tree Params: {dt_grid.best_params_}")
evaluate_model(best_dt, X_train, X_test, y_train, y_test)

Το μοντέλο του decision tree δεν συμπεριφέρεται τόσο καλά όσο της Logistic Regression. Η κυριότερη αδυναμία του είναι ότι υπερπροσαρμόζεται. Στο training set έχει $accuracy=87\%$ ενώ στο testing set μόλις $accuracy=72\%$. Επισης το εμβαδόν κάτω από την ROC είναι $0.72$ ενώ το $f_2score=77\%$. Το μοντέλο αυτό χάνει αρκετούς ασθενείς  και στο training set και στο testing set από ότι φαίνεται στους Confussion Matrices.

Αν θέλουμε να ερμηνεύσουμε το δέντρο μπορούμε να το οπτικοποιήσουμε: 

In [ ]:
from sklearn.tree import plot_tree

plt.figure(figsize=(20, 10)) # Μεγάλο μέγεθος για να φαίνεται
plot_tree(
    best_dt, 
    feature_names=X_train.columns, 
    class_names=['Healthy', 'Sick'], 
    filled=True, 
    rounded=True, 
    fontsize=10,
    max_depth=4 
)
plt.title("Οπτικοποίηση των Κανόνων του Δέντρου Απόφασης")
plt.show()

Εδώ η πρώτη ερώτηση που κάνει το μοντέλο-γιατρός για να κρίνει αν ένας ασθενής πάσχει από καρδιακή νόσο είναι αν έχει θαλασαιμία τύπου 2. Αν ναι τότε ρωτάει αν έχει στηθαγή από άσκηση (exang) διαφορετικά ρωτάει αν έχει μέγιστο καρδιακό παλμό μικρότερο των 137 κ.ο.κ. Παρατηρούμε πως εδώ τα γνωρίσματα έχουν διαφορετική "προτεραιότητα" αυτό συμβαίνει γιατί το δυαδικό δέντρο δεν προσπαθεί να χωρίσει τα δεδομένα γραμμικά και μπορεί να προβεί σε πιο πολύπλοκους συλλογισμούς.

### Random Forest

Εντελώς ανάλογα θα εργαστούμε και στο μοντέλο Random Forest. Eπειδή το dataset είναι αρκετά μικρό για ένα τόσο πολύπλοκο μοντέλο θα εισάγουμε περισσότερες παραμέτρους για να αποφύγουμε το overfitting. Στα Random Forests μπορούμε να διαλέξουμε και τον αριθμό των δέντρων που θέλουμε να συμπεριλάβουμε. Η παράμετρος min_samples_split δηλώνει τον ελάχιστο αριθμό ατόμων που πρέπει να έχουμε σε εναν κόμβο για να ξαναχωριστεί, ενώ η min_samples_leaf τον ελάχιστο αριθμό για κάθε τελικό κουτί. Η παράμετρος max_features δέχεται την πράξη που πρέπει να κάνει για να πάρει τον μέγιστο αριθμό χαρακτηριστικών (ή ποσοστό) που θα λάβει υπόψη: 

In [ ]:
rf_params = {'n_estimators': [200, 300], 'max_depth': [3, 4, 6],
            'min_samples_split': [ 3, 5, 8, ], 'min_samples_leaf': [4, 5, 9],      
            'max_features': ['sqrt', None, 0.5]}
rf_grid = GridSearchCV(RandomForestClassifier(random_state=42), rf_params, cv=5)
rf_grid.fit(X_train, y_train) 
best_rf = rf_grid.best_estimator_
print(f"\nBest Random Forest Params: {rf_grid.best_params_}")
evaluate_model(best_rf, X_train, X_test, y_train, y_test)

Και από τα Random Forests πήραμε ένα καλό μοντέλο. Αυτή τη φορά η ακρίβεια κυμαίνεται σε $accuracy=82\%$ ενώ το $precision_1=91\%$. Το μοντέλο δεν φαίνεται να πάσχει από overfitting αφού παρόμοια απόδοση έχει στο training set. Το $f_2score=78\%$ ενώ το εμβαδόν κάτω από την $ROC$ ισούται με $0.894$. Όλα αυτά δείχνουν μια καλή συμπεριφορά αλλά όπως φαίνεται και στον Confussion Πίνακα το μοντέλο δίνει αρκετά πιο πολλά ψευδώς αρνητικά αποτελέσματα από ψευδώς θετικά στο training set.


**Σημείωση:** Στα GridSearch για το tuning των παραμέτρων στα τελευταία δυο μοντέλα δεν δοκιμάστηκαν αυτές οι παράμετροι μόνο. Δοκιμάστηκαν πολλές ακόμη απλά αφαιρέθηκαν προκειμένου να μειωθεί ο απαραίτητος χρόνος εκτέλεσης των αντίστοιχων κελιών.

Δυστυχώς τα τυχαία δάση δεν είναι τόσο ερμηνεύσιμα όσο τα δυαδικά δεντρα. Παρόλα αυτά μπορούμε να ελέγξουμε ποια γνωρίσματα είναι πιο σημαντικά για το μοντέλο μας ως εξής:

In [ ]:
importances = best_rf.feature_importances_
feature_names = X_train.columns
rf_importances = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
})
rf_importances = rf_importances.sort_values(by='Importance', ascending=False)
print("Top 10 Σημαντικότερα Χαρακτηριστικά (Random Forest):")
print(rf_importances.head(10))
plt.figure(figsize=(10, 8))
sns.barplot(x='Importance', y='Feature', hue='Feature', data=rf_importances, palette='viridis', legend=False)
plt.title('Ποια χαρακτηριστικά επηρεάζουν περισσότερο το Random Forest;')
plt.xlabel('Βαθμός Σπουδαιότητας')
plt.ylabel('Χαρακτηριστικά')
plt.show()

Στην ουσία ο τρόπος με τον οποίο παίρνουμε τον βαθμό πουδαιότητας κάθε γνωρίσματος είναι μετρώντας τον μέσο όρο της σπουδαιότητας του κόμβου που φτιάχνει ένα χαρακτηριστικό σε όλα τα δέντρα που χρησιμοποιεί το δάσος. Έτσι αν ο διαχωρισμός με βάση ένα χαρακτηριστικό οδηγεί σε μεγάλη καθαρότητα σε πολλά δέντρα τότε το χαρακτηριστικό αυτό έχει μεγάλη σπουδαιότητα για το δάσος μας.

## Συμπέρασμα - Συγκριση Μοντέλων
Και στα 3 μοντέλα είδαμε πως σημαντικό ρόλο παίζουν τα γνωρίσματα thalalch & thal_2 ωστόσο η ιεραρχία των υπόλοιπων χαρακτηριστικών είναι αρκετά διαφορετική ειδικά σε σύγκριση με την Λογιστική Παλινδρόμηση (τα δενδροειδή μοντέλα έχουν αρκετές ομοιότητες μεταξύ τους). Για αυτό και οδηγούμαστε σε διαφορετικές αποδόσεις. 
Tα συμπεράσματα μας από την αξιολόγηση κάθε μοντέλου συνοψίζονται στον ακόλουθο πίνακα: 

| Μετρική | **Logistic Regression** | **Decision Tree** | **Random Forest** |
| :--- | :---: | :---: | :---: |
| **Accuracy** | 0.85 | 0.72 | 0.82 |
| **Precision** | 0.88 | 0.72 | 0.91|
| **Recall** | 0.85 | 0.79 | 0.76 |
| **F2-Score** | 0.85 | 0.77 | 0.78 |
| **AUC** | 0.90 | 0.73 | 0.89 |
|  **Overfitting**  | Όχι| Ναι | Όχι|

Όπως είπαμε και πριν προτιμάμε να έχουμε Ψευδώς Θετικές προβλέψεις παρά Ψευδώς Αρνητικές διατηρώντας βέβαια πάντα τις ψευδές προβλέψεις όσο το δυνατόν χαμηλότερα. Επομένως τα κριτήρια μας για να διαλέξουμε ποιο είναι καλύτερο μοντέλο για την περίπτωση μας θα πρέπει να βασίζεται στο Recall & Accuracy. Η προτίμηση μας εκφράζεται και από την μετρική του $F_2score=5\cdot \frac{Precision \cdot Recall}{4Precission + Recall}$. Για το παρόν dataset:
- Το μοντέλο της Logistic Regression είναι αυτό που έχει μεγαλύτερη ακρίβεια αλλά και F2-Score επομένως είναι το προτιμότερο για ιατρική χρήση.
- Ένα ακόμη πλεονέκτημα που έχει έναντι του Random Forest είναι ότι έχει μεγαλύτερη ερμηνευσιμότητα καθώς από τους συντελεστές μπορούμε να δούμε ποια γνωρίσματα παίζουν μεγαλύτερο ρόλο. Έτσι ενας γιατρός κοιτώντας το μοντέλο μπορεί να δώσει προτεραιότητα στα ενδεχόμενα συμπτώματα. 
- To απλό δυαδικό δέντρο είναι το πιο αναποτελεσματικό.